# Web Hosting

This example recreates the first CloudLab example where we launch a web server. I will also provide the example to enable 
SSH tunneling to access this server from the local computer. 

---

## Why Use SSH Tunnels?

Many FABRIC VMs are not directly reachable from the public Internet. Instead, access is typically provided through a **bastion host**. SSH tunnels allow you to forward ports securely through the bastion to reach internal services running on private VMs.

## Import the FABlib Library

In [2]:
from fabrictestbed_extensions.fablib.fablib import FablibManager as fablib_manager

fablib = fablib_manager()
fablib.get_image_names()

User: lngo@wcupa.edu bastion key is valid!
Configuration is valid


{'default_centos8_stream': {'description': 'CentOS 8 Stream (non-stream)',
  'default_user': 'centos'},
 'default_centos9_stream': {'description': 'CentOS 9 Stream (default install)',
  'default_user': 'cloud-user'},
 'default_centos10_stream': {'description': 'CentOS 10 Stream (default install)',
  'default_user': 'cloud-user'},
 'default_debian_11': {'description': 'Debian 11 Bullseye',
  'default_user': 'debian'},
 'default_debian_12': {'description': 'Debian 12 Bookworm',
  'default_user': 'debian'},
 'default_fedora_39': {'description': 'Fedora 39', 'default_user': 'fedora'},
 'default_fedora_40': {'description': 'Fedora 40', 'default_user': 'fedora'},
 'default_freebsd_13_zfs': {'description': 'FreeBSD 13 with ZFS',
  'default_user': 'freebsd'},
 'default_freebsd_14_zfs': {'description': 'FreeBSD 14 with ZFS',
  'default_user': 'freebsd'},
 'default_kali': {'description': 'Kali Linux (for penetration testing)',
  'default_user': 'kali'},
 'default_openbsd_7': {'description': 'Ope

## Create the Experiment Slice



In [3]:
slice_name = "web_ansible"

In [5]:
# Create a slice
slice = fablib.new_slice(name=slice_name)

# Add a node
node = slice.add_node(name="control_node", disk=10, image='default_ubuntu_22')

# Submit the slice
slice.submit();


Retry: 5, Time: 122 sec


ID,697d3cb2-a6a2-4225-939e-da68e4cb54f2
Name,web_ansible
Lease Expiration (UTC),2026-04-01 15:29:26 +0000
Lease Start (UTC),2026-03-31 15:29:26 +0000
Project ID,8b6dfc51-02ad-4f00-a389-6e75d8b61a26
State,StableOK
Email,lngo@wcupa.edu
UserId,8eecd713-fa8f-4b3b-8883-1ff9b021fa53


ID,Name,Cores,RAM,Disk,Image,Image Type,Host,Site,Username,Management IP,State,Error,SSH Command,Public SSH Key File,Private SSH Key File
24721ed2-f266-479d-a6da-47e42d3378cd,control_node,2,8,10,default_ubuntu_22,qcow2,mich-w2.fabric-testbed.net,MICH,ubuntu,None,Active,,ssh -i /home/fabric/.ssh/slice_key -F /home/fabric/work/fabric_config/ssh_config ubuntu@None,/home/fabric/.ssh/slice_key.pub,/home/fabric/.ssh/slice_key


In [8]:
import time

while True:
    time.sleep(10)
    slice.update()
    print("Slice state:", slice.get_state())
    print("Slice stable:", slice.isStable())
    if node.get_management_ip() != None:
        for node in slice.get_nodes():
            print("----", node.get_name(), "----")
            print("reservation state:", node.get_reservation_state())
            print("management ip:", node.get_management_ip())
            print("username:", node.get_username())
            print("error:", node.get_error_message())
            print(node.get_ssh_command())
        break

Slice state: StableOK
Slice stable: True
---- control_node ----
reservation state: Active
management ip: 2607:f018:110:11:f816:3eff:fee7:a9a9
username: ubuntu
error: 
ssh -i /home/fabric/.ssh/slice_key -F /home/fabric/work/fabric_config/ssh_config ubuntu@2607:f018:110:11:f816:3eff:fee7:a9a9


## Run a Web Service on the Node 

Test that the web service is up on the node slice

In [ ]:
slice = fablib.get_slice(slice_name)
node = slice.get_node('control_node')


node.upload_directory('ansible','.')
node.execute('bash ansible/setup_ansible.sh',  quiet=True, output_file=f"{node.get_name()}-ansible.log")

In [ ]:
node.execute("curl localhost:80", quiet=True, output_file=f"{node.get_name()}.log");

## Start the SSH Tunnel

- Create SSH Tunnel Configuration `fabric_ssh_tunnel_tools.zip`
- Download your custom `fabric_ssh_tunnel_tools.zip` tarball from the `fabric_config` folder.  
- Untar the tarball and put the resulting folder (`fabric_ssh_tunnel_tools`) somewhere you can access it from the command line.
- Open a terminal window. (Windows: use `powershell`) 
- Use `cd` to navigate to the `fabric_ssh_tunnel_tools` folder.
- Run the following command to setup permission correctly

```bash
chmod 600 slice_key fabric-bastion-key
```

- In your terminal, run the command that results from running the following cell (leave the terminal window open).

In [ ]:
fablib.create_ssh_tunnel_config(overwrite=True)

#### Port Forwarding Example

FileBrowser is running on the VM at port `5555`, use the following SSH command from your laptop:

In [ ]:
import os
# Port on your local machine that you want to map the web server to. This should be a port that you have specified on 
# docker-compose.yml

local_port='5555'
# We use 0.0.0.0 because we want the ability to forward this interface outside of the container. 
local_host='0.0.0.0'

# Port on the node used by the web server
target_port='80'

# Username/node on FABRIC
target_host=f'{node.get_username()}@{node.get_management_ip()}'

print(f'ssh  -L {local_host}:{local_port}:127.0.0.1:{target_port} -i {os.path.basename(fablib.get_default_slice_public_key_file())[:-4]} -F ssh_config {target_host}')

## Connect to the Web Server from your local browser

The browser service on the node is now mapped to 127.0.0.1:5555 on your local machine. You can open a browser and navigate to the following address (or just click the link): 

[http://127.0.0.1:5555](http://127.0.0.1:5555)

## Delete the Slice

Please delete your slice when you are done with your experiment.

In [ ]:
slice.delete()